# Lesson 2 Tokenization 

At the end of the day machine learning models are just doing math. This means that we need a way to turn words into a standardized set of numbers. This processes is called tokenization. 

## Tokenizers and Vocabularies

Tokenization happens in a few steps:

1. First we need to standardize our text and split them into tokens. At it's most basic this means splitting sentences into individual words and forcing text to lowercase
2. Then we can use a **Counter** object to categorize the words and give each of them a distinct integer value This mapping is called a **Vocabulary**. 
3. Using our vocabulary we create a second tokenizer to convert our  into nice tokenized input that our model can understand.


In [1]:
#First lets import our packages and redownload our dataset
import torch
from datasets import load_dataset
from collections import Counter
import re

#load our dataset and take a more managable slice to work with
nyt_dataset = load_dataset(path='ErikCikalleshi/new_york_times_news_1987_1995', split='test')
train_raw = nyt_dataset[:100]

c:\Users\ryanm\Comp Sci Work\Semantics-Research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Our train raw variable is a dictionary of lists instead of a Dataset
print(train_raw['date'])
print(train_raw['content'])

[1992, 1988, 1989, 1987, 1987, 1987, 1993, 1991, 1988, 1992, 1987, 1991, 1988, 1994, 1994, 1995, 1988, 1989, 1990, 1990, 1993, 1992, 1994, 1995, 1990, 1988, 1995, 1990, 1993, 1989, 1993, 1994, 1994, 1988, 1988, 1989, 1994, 1994, 1991, 1993, 1987, 1993, 1993, 1989, 1994, 1987, 1994, 1993, 1993, 1988, 1992, 1991, 1989, 1987, 1995, 1992, 1988, 1995, 1993, 1994, 1991, 1987, 1990, 1991, 1990, 1988, 1988, 1989, 1990, 1992, 1987, 1989, 1988, 1990, 1987, 1995, 1991, 1991, 1994, 1989, 1995, 1995, 1994, 1990, 1992, 1993, 1991, 1988, 1989, 1991, 1989, 1988, 1990, 1995, 1989, 1990, 1989, 1993, 1991, 1995]
["EVEN if your house is well insulated, unsealed cracks and other air leaks may be robbing you of comfort and raising your heating and cooling bills. Weatherstripping around doors and windows can help, but for the maximum benefit, spend a day or two scouting out and plugging leaks in less obvious places.\nTo detect air leaks in a building, professional energy auditors use instruments ranging from

Now we can create a basic tokenize function to standardize words and split sentence strings into lists of words.
* "Hello my name is Ryan" -> ["hello", "my", "name", "is", "ryan"]

In [4]:
def word_tokenize(text: str) -> list[str]:
    """Basic tokenizer: strip HTML tags, lowercase, split on whitespace."""
    text = text.lower()
    return text.split()

#TODO try tokenizing an article from the train_raw['content'] list. How does this format help us?



### Creating a Vocabulary

Now we can start creating our **Vocabulary**. A crucial part of this process is the **Counter class**. Counters are a versatile data structure that make language processing much easier. At their simplest they turn a list into a dictionary where each unique word is a key holding the number of instances as its value. 

Take a look at the Counter demo below. Try changing the sentence variable. 
* How do different sentences impact the output? 
* What is being counted and how is it being structured?

In [6]:
# TODO Try changing up this sentence variable to see how the counter changes
sentence = "This is a test sentence to show off the utility of counters hello hello hello"

word_tokenized_sentence = word_tokenize(sentence)

my_counter = Counter()

my_counter.update(word_tokenized_sentence)

print(my_counter)




Counter({'hello': 3, 'this': 1, 'is': 1, 'a': 1, 'test': 1, 'sentence': 1, 'to': 1, 'show': 1, 'off': 1, 'the': 1, 'utility': 1, 'of': 1, 'counters': 1})


The function below will takes in our train_raw dictionary and builds a vocabulary dictionary by:
1. Using a counter on the dataset to find all unique word tokens
2. Sort the counter words by number of occurances and add them to the vocabulary dictionary
    * Such that: vocab['the'] = 2 , vocab['a'] = 3

The goal here is every word in our dataset is assigned a unique integer value.


Note the special tokens that we add:
* < pad >
* < unk >

What might these be used to represent in a fully tokenized sentence? What else might we want to represent with special tokens?

In [20]:
# Build vocabulary from training data
def build_vocab(data, max_vocab=10000) -> dict[str, int]:
    """
    Count word frequencies and keep the top max_vocab words.
    Returns: word_to_idx dict
    """
    counter = Counter()


    for row in data['content']:
        counter.update(word_tokenize(row))

    # Special tokens, if we needed more we would add them here
    vocab = {'<pad>': 0, '<unk>': 1}

    #iterate from most common to least common words and add them to vocab
    for word, count in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)

    return vocab


### Finally we can put it all together and start tokenizing some data

The tokenize_and_numericalize() function takes a piece of text and turns it into a list of vocab indices:
1. First we use our word_tokenize() function to separate the sentence into words
2. Second we loop over each token in the sentence
    * for each token we pass it into our vocab dictionary and get back the proper index. If our vocab doesn't have that index we use the < unk > token's index to represent an unknown word.
3. Finally we store each collected index in a list and return it.

In [23]:


def tokenize_and_numericalize(text: str, vocab: dict[str, int], max_length=300):
    """Convert text string to list of integer indices."""
    tokens = word_tokenize(text)[:max_length]
    indices = []
    for t in tokens:
        indices.append(vocab.get(t, vocab['<unk>']))
    return indices

vocab = build_vocab(train_raw)
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"\nExample tokenization:")




Vocabulary size: 10000

Example tokenization:


In [ ]:
'''
    Try tokenizing a sentence of your own below. Either index into train_raw['content'] or write your own.
    Notice what happens if you pass the tokenizer a word it hasn't seen before.

    Remember... an index of 1 means an <unk> token. Meaning that word wasn't in the data we used to build our vocabulary
'''

#change this index to try tokenizing a different line
sample_text = train_raw['content'][0]
sample_indices = tokenize_and_numericalize(sample_text, vocab)
print(f"  Text (first 60 chars): '{sample_text[:60]}...'")
print(f"  Indices (first 10):    {sample_indices[:10]}")
print(f"  Sequence length:       {len(sample_indices)}")

  Text (first 60 chars): 'EVEN if your house is well insulated, unsealed cracks and ot...'
  Indices (first 10):    [99, 53, 202, 116, 10, 245, 4101, 4102, 1404, 6]
  Sequence length:       300


# In Conclusion

Tokenization and vocabularies are important building blocks in both machine learning and all of natural language processing. Representing words as structured numbers is helpful for a lot of different projects. While we used a small section of our dataset to create a 10,000 word vocabulary, different tokenization strategies may use vocabularies of 30,000-100,000+ to help a model learn very specialized uses of a wide variety of words and word parts.

In the next Lesson we will use a **Dataloader** to organize our tokenized data into batches which we can finally pass to a model as part of the training process.